# EpiGraph-AI: Epidemic Outbreak Prediction

A spatio-temporal deep learning model utilizing Graph Attention Networks (GAT) and LSTM for predicting disease outbreaks, incorporating textual data from health news via BioBERT.

## Pipeline Overview
1.  **Data Loading & EDA**: Load case counts, news headlines, and district connectivity.
2.  **BioBERT Encoding**: Extract feature embeddings from health news headlines.
3.  **Graph Construction**: Build adjacency matrices with self-loops.
4.  **Feature Engineering**: Standardize, PCA encode, and window sequence data.
5.  **Model Training**: Train EpiGraphModel (GAT + LSTM) using Huber Loss.
6.  **Evaluation & Prediction**: Assess performance via MAE, RMSE, MAPE and Outbreak Thresholds.

## 1. Setup & Dependencies

In [ ]:
!pip install -q torch-geometric transformers scikit-learn seaborn
print("All dependencies installed!")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, f1_score, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModel
from torch_geometric.nn import GATv2Conv
import warnings

warnings.filterwarnings('ignore')

# Configuration
DATA_DIR = "data"
CASES_FILE = os.path.join(DATA_DIR, "processed_cases.csv")
NEWS_FILE = os.path.join(DATA_DIR, "health_news.csv")
CONNECTIVITY_FILE = os.path.join(DATA_DIR, "connectivity.csv")

# Hyperparameters
WINDOW_SIZE = 7
HORIZON = 1
HIDDEN_DIM = 64
EPOCHS = 100
LR = 0.0005
WEIGHT_DECAY = 1e-4
PATIENCE = 15
PCA_DIM = 32
BATCH_SIZE = 8
DROPOUT = 0.4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Data Loading & EDA

In [ ]:
cases_df = pd.read_csv(CASES_FILE)
news_df = pd.read_csv(NEWS_FILE)
conn_df = pd.read_csv(CONNECTIVITY_FILE)

print("=== Cases Data ===")
print(f"Shape: {cases_df.shape}")
display(cases_df.head(2))

print("
=== News Data ===")
print(f"Shape: {news_df.shape}")
display(news_df.head(2))

print("
=== Connectivity Data ===")
print(f"Shape: {conn_df.shape}")
display(conn_df.head(2))

In [ ]:
# Exploratory Data Analysis & Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for district in cases_df['District'].unique():
    d = cases_df[cases_df['District'] == district]
    axes[0, 0].plot(range(len(d)), d['dengue'].values, label=district, alpha=0.7)
axes[0, 0].set_title('Dengue Cases Over Time by District')
axes[0, 0].set_xlabel('Week Index')
axes[0, 0].set_ylabel('Cases')
axes[0, 0].legend(fontsize=8)

axes[0, 1].hist(cases_df['dengue'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(cases_df['dengue'].median(), color='red', linestyle='--', label=f"Median: {cases_df['dengue'].median():.0f}")
axes[0, 1].set_title('Distribution of Dengue Cases')
axes[0, 1].set_xlabel('Cases')
axes[0, 1].legend()

feature_cols_eda = ['dengue', '.MMAX', '.MMIN', '..TMRF', '.RH -0830', '.RH -1730']
corr = cases_df[feature_cols_eda].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 0], vmin=-1, vmax=1)
axes[1, 0].set_title('Feature Correlation Matrix')

news_df['Type'].value_counts().plot(kind='bar', ax=axes[1, 1], color=['coral', 'skyblue'])
axes[1, 1].set_title('News Type Distribution')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. BioBERT Feature Encoding

In [ ]:
class BioBERTEncoder:
    def __init__(self, model_name="dmis-lab/biobert-v1.1"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

    def encode(self, texts, batch_size=32):
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            inputs = self.tokenizer(
                batch_texts, padding=True, truncation=True,
                return_tensors="pt", max_length=64
            )
            with torch.no_grad():
                outputs = self.model(**inputs)
                embeddings = outputs.last_hidden_state[:, 0, :]
                all_embeddings.append(embeddings)
        return torch.cat(all_embeddings, dim=0)

bert_encoder = BioBERTEncoder()
print("BioBERT loaded successfully!")

## 4. Graph Construction

In [ ]:
districts = sorted(cases_df['District'].unique())
node_mapping = {d: i for i, d in enumerate(districts)}

num_nodes = len(districts)
adj_matrix = np.zeros((num_nodes, num_nodes))
for _, row in conn_df.iterrows():
    src_idx = node_mapping.get(row['Source'])
    tgt_idx = node_mapping.get(row['Target'])
    if src_idx is not None and tgt_idx is not None:
        adj_matrix[src_idx, tgt_idx] = row['Weight']
        adj_matrix[tgt_idx, src_idx] = row['Weight'] 

np.fill_diagonal(adj_matrix, 1.0)
adj_tensor = torch.tensor(adj_matrix, dtype=torch.float32)
edge_index = adj_tensor.nonzero().t().contiguous().to(device)

plt.figure(figsize=(6, 5))
sns.heatmap(adj_matrix, annot=True, fmt='.1f', xticklabels=districts, yticklabels=districts, cmap='YlOrRd')
plt.title('District Connectivity (with Self-Loops)')
plt.tight_layout()
plt.show()

## 5. Spatio-Temporal Data Processing

In [ ]:
feature_cols = ['dengue', '.MMAX', '.MMIN', '..TMRF', '.RH -0830', '.RH -1730']
dates = sorted(cases_df['Date'].unique())
full_idx = pd.MultiIndex.from_product([dates, districts], names=['Date', 'District'])

cases_filtered = cases_df[['Date', 'District'] + feature_cols].copy()
cases_indexed = cases_filtered.set_index(['Date', 'District'])
cases_filled = cases_indexed.reindex(full_idx, fill_value=0)
news_grouped = news_df.groupby(['Date', 'District'])['Headline'].apply(lambda x: " ".join(x))

num_timesteps = len(dates)
emb_dim_raw = 768

raw_case_features = []
raw_embeddings = []
raw_targets = []

for t, date in enumerate(dates):
    day_data = cases_filled.loc[date].reindex(districts, fill_value=0)
    raw_case_features.append(day_data[feature_cols].values)
    
    day_embeddings = []
    for district in districts:
        if (date, district) in news_grouped:
            headline = news_grouped.loc[(date, district)]
            emb = bert_encoder.encode([headline]).squeeze(0).numpy()
        else:
            emb = np.zeros(emb_dim_raw)
        day_embeddings.append(emb)
    
    raw_embeddings.append(np.array(day_embeddings))
    raw_targets.append(day_data[['dengue']].values)

raw_case_features = np.array(raw_case_features)
raw_embeddings = np.array(raw_embeddings)
raw_targets = np.array(raw_targets)

print(f"Features shaped successfully. Timesteps: {num_timesteps}")

In [ ]:
# Normalization and PCA
T, N, F_case = raw_case_features.shape

case_scaler = StandardScaler()
case_scaled = case_scaler.fit_transform(raw_case_features.reshape(-1, F_case)).reshape(T, N, F_case)

emb_flat = raw_embeddings.reshape(T * N, emb_dim_raw)
pca = PCA(n_components=PCA_DIM)
emb_pca = pca.fit_transform(emb_flat).reshape(T, N, PCA_DIM)
emb_scaler = StandardScaler()
emb_pca_scaled = emb_scaler.fit_transform(emb_pca.reshape(-1, PCA_DIM)).reshape(T, N, PCA_DIM)

combined_features = np.concatenate([case_scaled, emb_pca_scaled], axis=2)

target_scaler = StandardScaler()
target_scaled = target_scaler.fit_transform(raw_targets.reshape(-1, 1)).reshape(T, N, 1)

x_tensor = torch.tensor(combined_features, dtype=torch.float32)
y_tensor = torch.tensor(target_scaled, dtype=torch.float32)

print(f"Final Input Tensor Dimensions: {x_tensor.shape}")

In [ ]:
def create_windows(x_tensor, y_tensor, window_size=7, horizon=1):
    X_out, Y_out = [], []
    for i in range(len(x_tensor) - window_size - horizon + 1):
        X_out.append(x_tensor[i : i + window_size])
        Y_out.append(y_tensor[i + window_size : i + window_size + horizon])
    return torch.stack(X_out), torch.stack(Y_out)

x_windows, y_windows = create_windows(x_tensor, y_tensor, WINDOW_SIZE, HORIZON)

n_total = len(x_windows)
train_end = int(n_total * 0.70)
val_end = int(n_total * 0.85)

x_train, y_train = x_windows[:train_end].to(device), y_windows[:train_end].to(device)
x_val, y_val = x_windows[train_end:val_end].to(device), y_windows[train_end:val_end].to(device)
x_test, y_test = x_windows[val_end:].to(device), y_windows[val_end:].to(device)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

## 6. Model Architecture: GAT + LSTM

In [ ]:
class EpiGraphModel(nn.Module):
    def __init__(self, num_nodes, input_dim, hidden_dim, output_dim=1, heads=2, dropout=0.4):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim

        self.input_proj = nn.Linear(input_dim, hidden_dim)
        
        # Spatial Encoder: GAT
        self.gat1 = GATv2Conv(input_dim, hidden_dim, heads=heads, dropout=dropout)
        self.bn1 = nn.BatchNorm1d(hidden_dim * heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        # Temporal Encoder: LSTM
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True, num_layers=2, dropout=dropout)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        b, t, n, f = x.size()
        spatial_features_seq = []
        
        for i in range(t):
            xt = x[:, i, :, :]
            batch_spatial = []
            for bi in range(b):
                h = self.gat1(xt[bi], edge_index)
                h = F.elu(self.bn1(h))
                h = self.dropout(h)
                h = self.gat2(h, edge_index)
                h = F.elu(self.bn2(h))
                h = h + self.input_proj(xt[bi])  # Residual connection
                batch_spatial.append(h)
            spatial_features_seq.append(torch.stack(batch_spatial))
        
        spatial_seq = torch.stack(spatial_features_seq, dim=1)
        spatial_flat = spatial_seq.view(b * n, t, self.hidden_dim)
        
        lstm_out, _ = self.lstm(spatial_flat)
        out = self.fc(lstm_out[:, -1, :])
        return out.view(b, n, -1)

num_features = x_tensor.shape[2]
model = EpiGraphModel(num_nodes, num_features, HIDDEN_DIM, dropout=DROPOUT).to(device)
print(f"Total Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Training Loop

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.HuberLoss(delta=1.0)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=7, factor=0.5)

train_losses, val_losses = [], []
best_val_loss = float('inf')
best_model_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = model(bx, edge_index)
        loss = criterion(out, by.squeeze(1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
    
    avg_train = total_train_loss / len(train_loader)
    train_losses.append(avg_train)
    
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for bx, by in val_loader:
            out = model(bx, edge_index)
            loss = criterion(out, by.squeeze(1))
            total_val_loss += loss.item()
            
    avg_val = total_val_loss / len(val_loader)
    val_losses.append(avg_val)
    scheduler.step(avg_val)
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        
    if (epoch + 1) % 5 == 0 or patience_counter == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")
        
    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "epigraph_model_v2.pth")

## 8. Evaluation

In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for bx, by in test_loader:
        out = model(bx, edge_index)
        all_preds.append(out.cpu())
        all_targets.append(by.squeeze(1).cpu())

preds = target_scaler.inverse_transform(torch.cat(all_preds).numpy().flatten().reshape(-1, 1)).flatten()
targets = target_scaler.inverse_transform(torch.cat(all_targets).numpy().flatten().reshape(-1, 1)).flatten()
preds = np.clip(preds, 0, None)

mae = mean_absolute_error(targets, preds)
rmse = np.sqrt(np.mean((targets - preds) ** 2))
r2 = r2_score(targets, preds)

print(f"Overall Metrics  |  MAE: {mae:.4f}   RMSE: {rmse:.4f}   R²: {r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(targets, preds, alpha=0.5, s=20, color='steelblue')
max_val = max(targets.max(), preds.max()) * 1.1
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2)
axes[0].set(xlabel='Actual Cases', ylabel='Predicted Cases', title='Predicted vs Actual (Test Set)')
axes[0].grid(True, alpha=0.3)

preds_all = torch.cat(all_preds).numpy()
targets_all = torch.cat(all_targets).numpy()
d0_preds = np.clip(target_scaler.inverse_transform(preds_all[:, 0, :]).flatten(), 0, None)
d0_targets = target_scaler.inverse_transform(targets_all[:, 0, :]).flatten()

axes[1].plot(d0_targets, label='Actual', marker='o', markersize=3)
axes[1].plot(d0_preds, label='Predicted', linestyle='--', marker='s', markersize=3)
axes[1].set(xlabel='Test Sample Index', ylabel='Cases', title=f'Time Series: {districts[0]}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.show()